# The Hessian

_Linear Algebra_

**The matrix of second derivatives. It tells you the curvature, and whether you're convex.**

The Hessian is the matrix of all second partial derivatives of a function of several inputs.

     Where the gradient gives slope (first derivative), the Hessian gives curvature (second derivative): how the slope itself bends, in every pair of directions.

---

This notebook is a practice scaffold for the **The Hessian** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

We approximate the Hessian of a small two-variable function with finite differences, then check the two properties that matter: it should be **symmetric**, and its **eigenvalues** tell us whether the function is convex. We build it in three steps: (1) define the function, (2) estimate the Hessian numerically, (3) read off convexity from the eigenvalues.

### Step 1 — Define the function

We use `f(x, y) = x² + x·y + 2·y²`, a smooth bowl whose exact Hessian is the constant matrix `[[2, 1], [1, 4]]`. Because this function is quadratic, the curvature is the same everywhere — which makes it a clean target to check our numerical estimate against.

In [ ]:
# f(x, y) = x^2 + x*y + 2*y^2  ->  true Hessian [[2, 1], [1, 4]]
def f(v):
    x, y = v
    return x**2 + x*y + 2*y**2

### Step 2 — Estimate the Hessian by finite differences

Each second partial derivative `H[i, j]` is approximated by nudging the input by a tiny step `h` along axes `i` and `j` in all four +/- combinations — the standard central finite-difference stencil. We loop over every pair of axes to fill in the full matrix, then confirm it comes out **symmetric** (mixed partials are equal for smooth functions).

In [ ]:
def hessian(f, v, h=1e-4):
    n = len(v)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            # unit steps of size h along axis i and axis j
            ei = np.zeros(n)
            ej = np.zeros(n)
            ei[i] = h
            ej[j] = h
            # central second-difference stencil for d^2 f / dx_i dx_j
            H[i, j] = (f(v+ei+ej) - f(v+ei-ej) - f(v-ei+ej) + f(v-ei-ej)) / (4*h*h)
    return H

H = hessian(f, np.array([1.0, 1.0]))
print("Hessian ~\n", np.round(H, 3))
print("symmetric:", np.allclose(H, H.T, atol=1e-3))

### Step 3 — Read convexity off the eigenvalues

A function is **convex** exactly when its Hessian is positive semi-definite, which means all of its eigenvalues are non-negative. We use `eigvalsh` (the symmetric-matrix solver) to get the eigenvalues; here both are positive, so the bowl curves upward in every direction and the function is convex.

In [ ]:
# convex <=> Hessian PSD <=> eigenvalues >= 0
eigenvalues = np.linalg.eigvalsh(H)
print("eigenvalues:", np.round(eigenvalues, 3), " -> convex:", np.all(eigenvalues > 0))

## Visualize the data & results

_How do the Hessian and its eigenvalues distinguish a convex bowl from a saddle?_

### Step 1 — Recompute the Hessian at the origin

For the plot we rebuild the same finite-difference Hessian, this time evaluated at the origin, and grab its eigenvalues. Since the function is quadratic the matrix is identical to before — `[[2, 1], [1, 4]]` — and both eigenvalues are positive.

In [ ]:
def f(v):
    x, y = v
    return x**2 + x*y + 2*y**2            # true Hessian [[2, 1], [1, 4]]

h = 1e-4
H = np.zeros((2, 2))
for i in range(2):                        # Hessian via finite differences
    for j in range(2):
        ei = np.zeros(2)
        ej = np.zeros(2)
        ei[i] = h
        ej[j] = h
        H[i, j] = (f(ei+ej) - f(ei-ej) - f(-ei+ej) + f(-ei-ej)) / (4*h*h)

w = np.linalg.eigvalsh(H)                  # both positive -> convex

### Step 2 — Draw the Hessian heatmap and the curvature curves

The left panel shows the Hessian as a labelled heatmap so you can see its entries directly. The right panel contrasts two parabolas built from the first eigenvalue: a positive coefficient gives the upward **convex bowl**, while flipping the sign gives the downward **saddle direction** you would see if an eigenvalue were negative.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

# Left — the Hessian as a labelled heatmap
ax1.imshow(H, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax1.text(j, i, str(round(H[i, j])), ha='center', va='center')
ax1.set_title('Hessian (finite differences)')
ax1.set_xticks([0, 1], ['d/dx', 'd/dy'])
ax1.set_yticks([0, 1], ['d/dx', 'd/dy'])

# Right — convex bowl vs saddle direction from the eigenvalue
t = np.linspace(-2, 2, 9)
ax2.plot(t, w[0]/2 * t**2, color='#7ee787', label='convex bowl (positive eig)')
ax2.plot(t, -w[0]/2 * t**2, color='#ff7b72', label='saddle direction')
ax2.set_xlabel('step along an axis')
ax2.set_ylabel('function value')
ax2.legend()

plt.tight_layout()
plt.show()